# 05 — Risk Scoring Model
**CamAgri Agricultural Risk Assessment Platform**

This notebook builds, calibrates and evaluates the agricultural risk scoring pipeline:
- **Financial risk** — derived from historical price volatility (CV)
- **Climate risk** — composite from drought index, flood risk, rainfall variability
- **Agronomic risk** — rule-based from soil pH, temperature, rainfall thresholds
- **Overall risk** — weighted blend, classified into low / medium / high / critical

**Sections**
1. Setup & data loading
2. Risk component analysis
3. Risk classifier training (3 tiers)
4. Model calibration
5. Scenario analysis
6. Risk map by region × crop
7. Model evaluation
8. Save models

## 1. Setup & Data Loading

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.ensemble import (
    GradientBoostingClassifier, RandomForestClassifier,
    GradientBoostingRegressor,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, RobustScaler, label_binarize
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
    brier_score_loss,
)
from scipy import stats

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('[INFO] SHAP not installed: pip install shap')

warnings.filterwarnings('ignore')

DATA_ROOT  = Path('../data')
FEAT_DIR   = Path('../outputs/features')
MODELS_DIR = Path('../models')
OUTPUT_DIR = Path('../outputs/risk')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor':'#0d0d0d','axes.facecolor':'#111111',
    'axes.edgecolor':'#333333','text.color':'#cccccc',
    'axes.labelcolor':'#cccccc','axes.titlecolor':'#ffffff',
    'xtick.color':'#666666','ytick.color':'#666666',
    'grid.color':'#222222','grid.alpha':0.5,'font.size':11,
})
LIME = '#CAFF00'
RISK_COLORS = {'low': LIME, 'medium': '#FFB800', 'high': '#FF6B00', 'critical': '#FF3B3B'}

CROPS   = ['maize','rice','cassava','cocoyam','plantain','cocoa','coffee',
           'groundnut','beans','tomato','onion','potato','palm_oil','sorghum']
REGIONS = ['North West','South West','West','Littoral','Adamawa',
           'Far North','Centre','East','North','South']

print('Setup complete.')

In [ ]:
# Load risk features dataset (from notebook 02)
try:
    risk_df = pd.read_parquet(FEAT_DIR / 'risk_features.parquet')
    print('Loaded engineered risk features.')
except FileNotFoundError:
    print('[ERROR] Run notebook 02 first to generate risk_features.parquet')
    raise

print(f'Shape: {risk_df.shape}')
print(f'Columns: {list(risk_df.columns)}')
print(f'\nRisk level distribution:')
print(risk_df['risk_level'].value_counts())

## 2. Risk Component Analysis

In [ ]:
# ── 2.1 Overall risk score distribution ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
ax = axes[0]
for level, c in RISK_COLORS.items():
    data = risk_df.loc[risk_df.risk_level == level, 'overall_risk']
    ax.hist(data, bins=30, alpha=0.65, color=c, label=level.capitalize(), edgecolor='none')
ax.set_title('Overall Risk Score Distribution by Level')
ax.set_xlabel('Overall Risk Score [0–1]')
ax.set_ylabel('Count')
ax.legend(fontsize=10)
ax.axvline(0.35, color='white', lw=1, linestyle='--', alpha=0.4)
ax.axvline(0.55, color='white', lw=1, linestyle='--', alpha=0.4)
ax.axvline(0.75, color='white', lw=1, linestyle='--', alpha=0.4)

# Pie
ax2 = axes[1]
level_counts = risk_df['risk_level'].value_counts()
ax2.pie(
    level_counts.values,
    labels=[l.capitalize() for l in level_counts.index],
    colors=[RISK_COLORS.get(l, '#888888') for l in level_counts.index],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': '#0d0d0d', 'linewidth': 2},
    textprops={'color': '#cccccc'},
)
ax2.set_title('Risk Level Proportions')

plt.tight_layout()
plt.savefig(OUTPUT_DIR/'risk_distribution.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.2 Risk components correlation ──────────────────────────────────────
risk_components = ['financial_risk_blended', 'climate_risk', 'agronomic_risk', 'overall_risk']
risk_components = [c for c in risk_components if c in risk_df.columns]

corr = risk_df[risk_components].corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Heatmap
sns.heatmap(
    corr, ax=axes[0], annot=True, fmt='.3f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.4, linecolor='#1a1a1a', annot_kws={'size': 11},
)
axes[0].set_title('Risk Component Correlations')

# Scatter: financial vs climate risk, coloured by level
if 'financial_risk_blended' in risk_df.columns and 'climate_risk' in risk_df.columns:
    sample = risk_df.sample(min(1500, len(risk_df)), random_state=42)
    for level, c in RISK_COLORS.items():
        mask = sample.risk_level == level
        axes[1].scatter(
            sample.loc[mask, 'financial_risk_blended'],
            sample.loc[mask, 'climate_risk'],
            c=c, alpha=0.5, s=14, edgecolors='none', label=level.capitalize(),
        )
    axes[1].set_xlabel('Financial Risk'); axes[1].set_ylabel('Climate Risk')
    axes[1].set_title('Financial vs Climate Risk')
    axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR/'risk_component_correlation.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.3 Risk by crop ──────────────────────────────────────────────────────
if 'crop' in risk_df.columns:
    crop_risk = (
        risk_df.groupby('crop')['overall_risk']
        .agg(mean='mean', median='median',
             p75=lambda x: x.quantile(.75),
             p90=lambda x: x.quantile(.90))
        .sort_values('mean', ascending=False)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(crop_risk))
    w = 0.3
    ax.bar(x - w/2, crop_risk['mean'],   w, label='Mean',   color='#444444', edgecolor='none')
    ax.bar(x + w/2, crop_risk['median'], w, label='Median', color=LIME,      edgecolor='none', alpha=0.8)
    ax.errorbar(x - w/2, crop_risk['mean'],
                yerr=crop_risk['p90']-crop_risk['mean'],
                fmt='none', color='white', capsize=4, lw=1)
    ax.set_xticks(x); ax.set_xticklabels(crop_risk.index, rotation=38, ha='right')
    ax.set_ylabel('Overall Risk Score')
    ax.set_title('Overall Risk Score by Crop (mean ± P90)')
    ax.axhline(0.55, color='#FF6B00', lw=1.5, linestyle='--', label='High risk threshold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'risk_by_crop.png', dpi=140, bbox_inches='tight')
    plt.show()

    print('\nTop 5 riskiest crops (by mean overall risk):')
    print(crop_risk['mean'].head(5).round(4))

In [ ]:
# ── 2.4 Risk by region ────────────────────────────────────────────────────
if 'region' in risk_df.columns:
    region_risk = (
        risk_df.groupby('region')[['overall_risk','climate_risk',
                                    'financial_risk_blended','agronomic_risk']]
        .mean()
        .sort_values('overall_risk', ascending=False)
        .round(4)
    )

    fig, ax = plt.subplots(figsize=(13, 6))
    x = np.arange(len(region_risk))
    w = 0.22
    components = [
        ('climate_risk',            '#38B6FF', 'Climate'),
        ('financial_risk_blended',  '#FFB800', 'Financial'),
        ('agronomic_risk',          LIME,      'Agronomic'),
    ]
    for i, (col, c, lbl) in enumerate(components):
        if col in region_risk.columns:
            ax.bar(x + (i-1)*w, region_risk[col], w, label=lbl, color=c, alpha=0.8, edgecolor='none')

    ax.plot(x, region_risk['overall_risk'], 'o-', color='white', lw=2, markersize=6, label='Overall', zorder=5)
    ax.set_xticks(x); ax.set_xticklabels(region_risk.index, rotation=35, ha='right')
    ax.set_ylabel('Risk Score'); ax.set_title('Risk Components by Region')
    ax.legend(fontsize=9, ncol=2); ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'risk_by_region.png', dpi=140, bbox_inches='tight')
    plt.show()

    print('\nRisk summary by region:')
    print(region_risk.to_string())

## 3. Risk Classifier Training (3 Tiers)

In [ ]:
# ── 3.1 Feature matrix ────────────────────────────────────────────────────
RISK_FEATURES = [
    'financial_risk_blended', 'climate_risk', 'agronomic_risk',
    'soil_ph', 'rainfall_mm', 'temperature_c',
]
RISK_FEATURES = [c for c in RISK_FEATURES if c in risk_df.columns]

RISK_LEVELS = ['low', 'medium', 'high', 'critical']
le_risk = LabelEncoder().fit(RISK_LEVELS)

df_risk = risk_df[RISK_FEATURES + ['risk_level']].dropna()
df_risk = df_risk[df_risk['risk_level'].isin(RISK_LEVELS)]

X_risk = df_risk[RISK_FEATURES].values
y_risk = le_risk.transform(df_risk['risk_level'])

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_risk, y_risk, test_size=0.20, stratify=y_risk, random_state=42
)

scaler_risk = RobustScaler()
X_tr_r_sc = scaler_risk.fit_transform(X_tr_r)
X_te_r_sc = scaler_risk.transform(X_te_r)
joblib.dump(scaler_risk, MODELS_DIR / 'risk_scaler.pkl')

print(f'Risk features ({len(RISK_FEATURES)}): {RISK_FEATURES}')
print(f'Train: {X_tr_r.shape}  |  Test: {X_te_r.shape}')
print(f'Classes: {le_risk.classes_}')

In [ ]:
# ── 3.2 Train tier models ─────────────────────────────────────────────────
risk_models = {
    'free':    DecisionTreeClassifier(
                   max_depth=6, min_samples_split=15,
                   class_weight='balanced', random_state=42),
    'medium':  RandomForestClassifier(
                   n_estimators=200, max_depth=12,
                   class_weight='balanced', random_state=42, n_jobs=-1),
    'premium': GradientBoostingClassifier(
                   n_estimators=300, learning_rate=0.05, max_depth=5,
                   subsample=0.8, random_state=42),
}

risk_results = {}
print('=== RISK CLASSIFIER TRAINING ===')

for tier, model in risk_models.items():
    model.fit(X_tr_r_sc, y_tr_r)
    y_pred = model.predict(X_te_r_sc)

    acc    = accuracy_score(y_te_r, y_pred)
    f1_mac = f1_score(y_te_r, y_pred, average='macro',    zero_division=0)
    f1_wtd = f1_score(y_te_r, y_pred, average='weighted', zero_division=0)

    risk_results[tier] = {
        'model':       model,
        'y_pred':      y_pred,
        'accuracy':    round(acc, 4),
        'f1_macro':    round(f1_mac, 4),
        'f1_weighted': round(f1_wtd, 4),
    }
    print(f'  [{tier:8s}]  Accuracy={acc:.4f}  '
          f'F1-macro={f1_mac:.4f}  F1-weighted={f1_wtd:.4f}')

In [ ]:
# ── 3.3 Detailed classification report — premium model ────────────────────
best = 'premium'
print(f'=== CLASSIFICATION REPORT — {best.upper()} TIER ===')
print(classification_report(
    y_te_r, risk_results[best]['y_pred'],
    target_names=le_risk.classes_, zero_division=0
))

In [ ]:
# ── 3.4 Confusion matrices (all tiers) ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (tier, res) in zip(axes, risk_results.items()):
    cm = confusion_matrix(y_te_r, res['y_pred'], normalize='true')
    sns.heatmap(
        cm, ax=ax, annot=True, fmt='.2f', cmap='YlGn',
        xticklabels=le_risk.classes_, yticklabels=le_risk.classes_,
        linewidths=0.3, linecolor='#1a1a1a', annot_kws={'size': 9},
    )
    ax.set_title(f'{tier.capitalize()} — Acc={res["accuracy"]:.3f}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

fig.suptitle('Risk Level Confusion Matrices by Tier', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'risk_confusion_matrices.png', dpi=140, bbox_inches='tight')
plt.show()

## 4. Model Calibration

In [ ]:
# ── 4.1 Probability calibration — premium model ───────────────────────────
# Risk level probabilities must be well-calibrated for the API to return
# reliable confidence scores (not just argmax)

gbm_risk = risk_results['premium']['model']

# Calibrate with Platt scaling (sigmoid)
calibrated_risk = CalibratedClassifierCV(
    GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, random_state=42,
    ),
    method='sigmoid', cv=5,
)
calibrated_risk.fit(X_tr_r_sc, y_tr_r)

# Check calibration per class
y_prob_raw  = gbm_risk.predict_proba(X_te_r_sc)
y_prob_cal  = calibrated_risk.predict_proba(X_te_r_sc)
y_binary    = label_binarize(y_te_r, classes=np.arange(len(RISK_LEVELS)))

fig, axes = plt.subplots(1, len(RISK_LEVELS), figsize=(18, 5))
for i, (ax, level) in enumerate(zip(axes, le_risk.classes_)):
    if y_binary[:, i].sum() == 0:
        ax.set_visible(False); continue

    frac_pos_raw, mean_pred_raw = calibration_curve(
        y_binary[:, i], y_prob_raw[:, i], n_bins=8, strategy='uniform'
    )
    frac_pos_cal, mean_pred_cal = calibration_curve(
        y_binary[:, i], y_prob_cal[:, i], n_bins=8, strategy='uniform'
    )

    ax.plot([0,1], [0,1], 'white', lw=1, linestyle='--', label='Perfect')
    ax.plot(mean_pred_raw, frac_pos_raw, 's-', color='#444444', lw=1.8, label='Uncalibrated')
    ax.plot(mean_pred_cal, frac_pos_cal, 'o-', color=RISK_COLORS[level], lw=2.5, label='Calibrated')
    ax.set_title(f'{level.capitalize()} risk')
    ax.set_xlabel('Mean predicted prob'); ax.set_ylabel('Fraction of positives')
    ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1)

fig.suptitle('Calibration Curves — Risk Level Probabilities (Premium GBM)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'risk_calibration_curves.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.2 Brier scores ──────────────────────────────────────────────────────
print('=== BRIER SCORES (lower = better probability calibration) ===')
for i, level in enumerate(le_risk.classes_):
    if y_binary[:, i].sum() == 0:
        continue
    bs_raw = brier_score_loss(y_binary[:, i], y_prob_raw[:, i])
    bs_cal = brier_score_loss(y_binary[:, i], y_prob_cal[:, i])
    print(f'  {level:10s}  raw={bs_raw:.4f}  calibrated={bs_cal:.4f}'
          f'  ({"improved" if bs_cal < bs_raw else "worsened"} by {abs(bs_cal-bs_raw)*100:.2f}pp)')

## 5. Scenario Analysis

In [ ]:
# ── 5.1 Define scenarios ──────────────────────────────────────────────────
# Vary one factor at a time to understand risk sensitivity

def score_scenario(conditions: dict, model=None, scaler=None) -> dict:
    """Returns risk scores for a given condition vector."""
    if model is None:  model  = calibrated_risk
    if scaler is None: scaler = scaler_risk

    vec   = np.array([[conditions.get(f, 0.0) for f in RISK_FEATURES]])
    vec_sc = scaler.transform(vec)
    proba  = model.predict_proba(vec_sc)[0]
    pred   = le_risk.classes_[proba.argmax()]
    return {
        'risk_level': pred,
        'probabilities': dict(zip(le_risk.classes_, proba.round(4))),
        'overall_score': round(proba @ np.array([0.15, 0.40, 0.70, 0.95]), 4),
    }

# Base scenario
BASE = {
    'financial_risk_blended': 0.45,
    'climate_risk':           0.40,
    'agronomic_risk':         0.30,
    'soil_ph':                6.2,
    'rainfall_mm':            1600,
    'temperature_c':          25.0,
}

base_result = score_scenario(BASE)
print('Base scenario result:', base_result)

In [ ]:
# ── 5.2 Rainfall sensitivity analysis ────────────────────────────────────
rainfall_vals  = np.linspace(200, 4000, 50)
rainfall_scores = []
for rain in rainfall_vals:
    sc = dict(BASE); sc['rainfall_mm'] = rain
    r  = score_scenario(sc)
    rainfall_scores.append(r['overall_score'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rainfall_vals, rainfall_scores, color=LIME, lw=2.5)
ax.fill_between(rainfall_vals, 0, rainfall_scores, alpha=0.12, color=LIME)

# Shade risk zones
ax.axhline(0.35, color='#CAFF00', lw=1, linestyle=':', alpha=0.5)
ax.axhline(0.55, color='#FFB800', lw=1, linestyle=':', alpha=0.5)
ax.axhline(0.75, color='#FF6B00', lw=1, linestyle=':', alpha=0.5)
for ylo, yhi, c, lbl in [(0,.35,'#CAFF00','Low'),(0.35,.55,'#FFB800','Medium'),
                           (0.55,.75,'#FF6B00','High'),(0.75,1.0,'#FF3B3B','Critical')]:
    ax.axhspan(ylo, yhi, alpha=0.04, color=c)
    ax.text(3950, (ylo+yhi)/2, lbl, va='center', ha='right', fontsize=9, color=c)

ax.set_xlabel('Annual Rainfall (mm)')
ax.set_ylabel('Overall Risk Score')
ax.set_title('Risk Score Sensitivity to Rainfall — all other factors held at base')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'risk_rainfall_sensitivity.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.3 Multi-variable sensitivity heatmap ───────────────────────────────
rain_vals   = np.linspace(300,  3500, 20)
finrisk_vals= np.linspace(0.1,  0.9,  20)

heat = np.zeros((len(finrisk_vals), len(rain_vals)))
for i, fr in enumerate(finrisk_vals):
    for j, rain in enumerate(rain_vals):
        sc = dict(BASE)
        sc['rainfall_mm']            = rain
        sc['financial_risk_blended'] = fr
        heat[i, j] = score_scenario(sc)['overall_score']

fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(
    heat, aspect='auto', origin='lower', cmap='RdYlGn_r',
    extent=[rain_vals[0], rain_vals[-1], finrisk_vals[0], finrisk_vals[-1]],
    vmin=0, vmax=1,
)
plt.colorbar(im, ax=ax, label='Overall Risk Score')
ax.set_xlabel('Annual Rainfall (mm)')
ax.set_ylabel('Financial Risk Score')
ax.set_title('Risk Score Heatmap — Rainfall × Financial Risk')

# Add contour lines at risk thresholds
cs = ax.contour(rain_vals, finrisk_vals, heat,
                levels=[0.35, 0.55, 0.75],
                colors=['#CAFF00','#FFB800','#FF3B3B'], linewidths=1.5)
ax.clabel(cs, inline=True, fontsize=9,
           fmt={0.35:'Low/Medium', 0.55:'Medium/High', 0.75:'High/Critical'})

plt.tight_layout()
plt.savefig(OUTPUT_DIR/'risk_sensitivity_heatmap.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.4 Named scenario comparison ────────────────────────────────────────
scenarios = {
    'Baseline (Centre, maize, moderate market)': {
        'financial_risk_blended':0.45,'climate_risk':0.40,'agronomic_risk':0.30,
        'soil_ph':6.2,'rainfall_mm':1600,'temperature_c':25.0
    },
    'Far North, tomato, poor market': {
        'financial_risk_blended':0.82,'climate_risk':0.85,'agronomic_risk':0.62,
        'soil_ph':6.8,'rainfall_mm':450, 'temperature_c':32.0
    },
    'South West, palm oil, good market': {
        'financial_risk_blended':0.30,'climate_risk':0.35,'agronomic_risk':0.15,
        'soil_ph':5.8,'rainfall_mm':3200,'temperature_c':26.0
    },
    'Adamawa, coffee, moderate market': {
        'financial_risk_blended':0.48,'climate_risk':0.62,'agronomic_risk':0.35,
        'soil_ph':6.0,'rainfall_mm':1100,'temperature_c':22.0
    },
    'Littoral, rice, excellent market': {
        'financial_risk_blended':0.28,'climate_risk':0.42,'agronomic_risk':0.20,
        'soil_ph':6.5,'rainfall_mm':2600,'temperature_c':27.0
    },
}

print('=== NAMED SCENARIO ANALYSIS ===')
scenario_results = {}
for name, cond in scenarios.items():
    res = score_scenario(cond)
    scenario_results[name] = res
    level_c = RISK_COLORS.get(res['risk_level'], '#888888')
    print(f'\n  {name}')
    print(f'    → Risk level: {res["risk_level"].upper():10s}  Overall: {res["overall_score"]:.4f}')
    print(f'    → Probabilities: {res["probabilities"]}')

## 6. Risk Map by Region × Crop

In [ ]:
# ── 6.1 Build region × crop risk matrix ──────────────────────────────────
FIN_RISK_MAP = {
    'tomato':0.82,'onion':0.75,'plantain':0.65,'cassava':0.40,'cocoa':0.60,
    'coffee':0.58,'maize':0.45,'rice':0.50,'cocoyam':0.42,'groundnut':0.55,
    'beans':0.48,'potato':0.60,'palm_oil':0.45,'sorghum':0.38,
}
CLIMATE_RISK_MAP = {
    'North West':0.45,'South West':0.35,'West':0.38,'Littoral':0.42,
    'Adamawa':0.62,'Far North':0.85,'Centre':0.40,'East':0.38,'North':0.78,'South':0.35,
}
MARKET_BASE = 0.55  # 'moderate' market access

# Build matrix
matrix = np.zeros((len(REGIONS), len(CROPS)))

for i, region in enumerate(REGIONS):
    for j, crop in enumerate(CROPS):
        cond = {
            'financial_risk_blended': (FIN_RISK_MAP[crop] + MARKET_BASE) / 2,
            'climate_risk':           CLIMATE_RISK_MAP[region],
            'agronomic_risk':         0.25,  # neutral agronomic baseline
            'soil_ph':                6.2,
            'rainfall_mm':            1500,
            'temperature_c':          25.0,
        }
        matrix[i, j] = score_scenario(cond)['overall_score']

risk_matrix_df = pd.DataFrame(matrix, index=REGIONS, columns=CROPS)

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(
    risk_matrix_df, ax=ax, cmap='RdYlGn_r',
    vmin=0.2, vmax=0.85,
    annot=True, fmt='.2f',
    linewidths=0.3, linecolor='#0d0d0d',
    annot_kws={'size': 8},
    cbar_kws={'label': 'Overall Risk Score'},
)
ax.set_title('Agricultural Risk Matrix — Region × Crop (moderate market access)', fontsize=14)
ax.set_xlabel('Crop'); ax.set_ylabel('Region')
ax.tick_params(axis='x', rotation=38)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'risk_region_crop_matrix.png', dpi=140, bbox_inches='tight')
plt.show()

print('\nHighest risk combinations:')
stacked = risk_matrix_df.stack().sort_values(ascending=False)
print(stacked.head(8).round(4))
print('\nLowest risk combinations:')
print(stacked.tail(8).round(4))

## 7. Model Evaluation

In [ ]:
# ── 7.1 ROC curves (one-vs-rest, premium model) ───────────────────────────
y_prob_prem = calibrated_risk.predict_proba(X_te_r_sc)
y_bin       = label_binarize(y_te_r, classes=np.arange(len(RISK_LEVELS)))

fig, ax = plt.subplots(figsize=(9, 7))
for i, (level, c) in enumerate(zip(le_risk.classes_, RISK_COLORS.values())):
    if y_bin[:, i].sum() < 5: continue
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob_prem[:, i])
    auc = roc_auc_score(y_bin[:, i], y_prob_prem[:, i])
    ax.plot(fpr, tpr, color=c, lw=2.5, label=f'{level.capitalize()} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'--',color='#555555',lw=1,label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Risk Level Classifier (One-vs-Rest)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'risk_roc_curves.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.2 SHAP feature importance — premium risk model ─────────────────────
if SHAP_AVAILABLE:
    # Use the base GBM (non-calibrated) for SHAP
    gbm_base = risk_results['premium']['model']
    X_shap_r = X_te_r_sc[:400]

    explainer_r = shap.TreeExplainer(gbm_base)
    shap_vals_r = explainer_r.shap_values(X_shap_r)

    # Average absolute SHAP across all classes
    if isinstance(shap_vals_r, list):
        mean_shap_r = np.mean([np.abs(sv) for sv in shap_vals_r], axis=0).mean(0)
    else:
        mean_shap_r = np.abs(shap_vals_r).mean(0)

    shap_feat_r = pd.Series(mean_shap_r, index=RISK_FEATURES).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, max(4, len(shap_feat_r)*0.55)))
    colors  = [LIME if v > shap_feat_r.median() else '#444444' for v in shap_feat_r.values]
    ax.barh(shap_feat_r.index, shap_feat_r.values, color=colors, edgecolor='none', height=0.6)
    ax.set_title('Mean |SHAP| — Risk Model (Premium GBM)')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'risk_shap.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    # Fallback: RF feature importance
    rf_risk = risk_results['medium']['model']
    fi_risk = pd.Series(rf_risk.feature_importances_, index=RISK_FEATURES).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(4, len(fi_risk)*0.55)))
    colors  = [LIME if v > fi_risk.median() else '#444444' for v in fi_risk.values]
    ax.barh(fi_risk.index, fi_risk.values, color=colors, edgecolor='none', height=0.6)
    ax.set_title('RF Feature Importance — Risk Model (Medium)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'risk_feature_importance.png', dpi=140, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 7.3 Cross-validation final check ──────────────────────────────────────
print('=== 5-FOLD STRATIFIED CV — RISK MODELS ===')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for tier, model in risk_models.items():
    cv_acc = cross_val_score(model, X_tr_r_sc, y_tr_r, cv=skf, scoring='accuracy', n_jobs=-1)
    cv_f1  = cross_val_score(model, X_tr_r_sc, y_tr_r, cv=skf, scoring='f1_macro', n_jobs=-1)
    print(f'  [{tier:8s}]  Accuracy={cv_acc.mean():.4f} ± {cv_acc.std():.4f}'
          f'  F1-macro={cv_f1.mean():.4f} ± {cv_f1.std():.4f}')

## 8. Save Models

In [ ]:
# ── 8.1 Save all risk models ──────────────────────────────────────────────
for tier, res in risk_results.items():
    path = MODELS_DIR / f'risk_{tier}.pkl'
    joblib.dump(res['model'], path)
    print(f'Saved: {path.name}')

# Save calibrated model separately
joblib.dump(calibrated_risk, MODELS_DIR / 'risk_premium_calibrated.pkl')
print('Saved: risk_premium_calibrated.pkl')

# Encoders & scalers
joblib.dump(le_risk,     MODELS_DIR / 'le_risk.pkl')
joblib.dump(scaler_risk, MODELS_DIR / 'risk_scaler.pkl')
print('Saved: le_risk.pkl, risk_scaler.pkl')

# Risk matrices for fast lookup
joblib.dump({
    'region_crop_matrix': risk_matrix_df,
    'fin_risk_map':        FIN_RISK_MAP,
    'climate_risk_map':    CLIMATE_RISK_MAP,
}, MODELS_DIR / 'risk_lookup_tables.pkl')
print('Saved: risk_lookup_tables.pkl')

# Update feature column registry
try:
    feat_cols = joblib.load(MODELS_DIR / 'feature_cols.pkl')
except FileNotFoundError:
    feat_cols = {}
feat_cols['risk'] = RISK_FEATURES
joblib.dump(feat_cols, MODELS_DIR / 'feature_cols.pkl')

print('\n=== ALL RISK MODELS SAVED ===')

In [ ]:
# ── 8.2 Final summary ─────────────────────────────────────────────────────
print('=' * 65)
print('  RISK SCORING — NOTEBOOK SUMMARY')
print('=' * 65)
print()
print('  MODEL PERFORMANCE:')
for tier, res in risk_results.items():
    print(f'    [{tier:8s}]  Accuracy={res["accuracy"]:.4f}  '
          f'F1-macro={res["f1_macro"]:.4f}')

print()
print('  SCENARIO INSIGHTS:')
for name, res in scenario_results.items():
    print(f'    {name[:50]:52s}  → {res["risk_level"].upper()} ({res["overall_score"]:.3f})')

print()
print('  RISK MATRIX HIGHLIGHTS:')
top3 = stacked.head(3)
for (region, crop), score in top3.items():
    print(f'    ⚠  {region:15s} × {crop:12s}  → {score:.3f}')
print()
bot3 = stacked.tail(3)
for (region, crop), score in bot3.items():
    print(f'    ✓  {region:15s} × {crop:12s}  → {score:.3f}')

print()
print('  Saved artifacts:')
for f in sorted(MODELS_DIR.glob('risk*.pkl')):
    print(f'    {f.name:45s}  {f.stat().st_size/1024:.1f} KB')
print('=' * 65)